In [ ]:
# %%
import pandas as pd
from core.paths import GLOBAL_DATA_DIR, GLOBAL_PROCESSED_DIR, LOCAL_DATA_DIR

DATE = "2026-08-14"
VIX_CLOSE = 14.25
VIX3M_CLOSE = 18.46

# 1. Load Data (without extra reset_index)
indices_path = GLOBAL_DATA_DIR / "VIX3M_VIX.parquet"
if indices_path.exists():
    df_indices = pd.read_parquet(indices_path)
    # Safety cleanup in case an 'index' column was saved previously
    if "index" in df_indices.columns:
        df_indices = df_indices.drop(columns=["index"])
else:
    raise FileNotFoundError(f"File not found: {indices_path}")

print("--- Before Appending ---")
print(df_indices.groupby("Ticker").tail(3))
print("=" * 60)


# 2. Reusable function to append new rows (OHLC defaults to close_p)
def append_index_data(
    df,
    ticker,
    date,
    close_p,
    open_p=None,
    high_p=None,
    low_p=None,
    volume=0.0,
):
    """Creates a new row and appends it to the DataFrame."""
    open_val = float(open_p if open_p is not None else close_p)
    high_val = float(high_p if high_p is not None else close_p)
    low_val = float(low_p if low_p is not None else close_p)
    close_val = float(close_p)

    new_data = pd.DataFrame(
        [
            {
                "Ticker": ticker,
                "Date": pd.to_datetime(date),
                "Adj Open": open_val,
                "Adj High": high_val,
                "Adj Low": low_val,
                "Adj Close": close_val,
                "Volume": float(volume),
            }
        ]
    )
    return pd.concat([df, new_data], ignore_index=True)


# 3. Append new daily data
df_indices = append_index_data(
    df=df_indices,
    ticker="^VIX",
    date=DATE,
    close_p=VIX_CLOSE,
    volume=0.0,
)

df_indices = append_index_data(
    df=df_indices,
    ticker="^VIX3M",
    date=DATE,
    close_p=VIX3M_CLOSE,
    volume=0.0,
)

# 4. Sort to keep tickers grouped and chronologically ordered
df_indices = df_indices.sort_values(by=["Ticker", "Date"]).reset_index(drop=True)

# 5. Save back to disk
df_indices.to_parquet(indices_path, index=False)

# 6. Verify final state
print("--- After Appending & Saved ---")
print(df_indices.groupby("Ticker").tail(3))